In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt

from selenium.common.exceptions import TimeoutException
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from scipy.stats import gaussian_kde
import numpy as np

# Function to extract match data from a single page
def extract_matches(soup):
    data = []
    for sublist in soup.find_all("div", class_="results-sublist"):
        for result_con in sublist.find_all("div", class_="result-con"):
            for a_reset in result_con.find_all("a", class_="a-reset"):
                result = a_reset.find("div", class_="result")

                # Extract team names and score
                team1 = result.find_all("td", class_="team-cell")[0].text.strip()
                score = result.find("td", class_="result-score").text.strip()
                team2 = result.find_all("td", class_="team-cell")[1].text.strip()

                # Create a dictionary for each match with relevant data
                match_data = {
                    "Team 1": team1,
                    "Team 2": team2,
                    "Score": score,
                }
                data.append(match_data)

    return data

# Set up the WebDriver (without specifying the driver path)
driver = webdriver.Firefox()

# Navigate to the first page
url = "https://www.hltv.org/results"
driver.get(url)

# Wait for the cookie banner to appear (up to 10 seconds) and click it
try:
    wait = WebDriverWait(driver, 10)
    cookie_banner = wait.until(EC.presence_of_element_located((By.XPATH, "//*[@id='CybotCookiebotDialogBodyButtonDecline']")))
    cookie_banner.click()
except TimeoutException:
    print("Cookie banner not found. Continuing with scraping...")

# Scrape match history for a specified number of pages
num_pages = 300  # Change this value to the desired number of pages
match_history = []

for page in range(1, num_pages + 1):
    offset = (page - 1) * 100
    url = f"https://www.hltv.org/results?offset={offset}"

    driver.get(url)

    # Get the HTML content
    html_content = driver.page_source

    # Parse the HTML content with BeautifulSoup
    soup = BeautifulSoup(html_content, "html.parser")

    match_history.extend(extract_matches(soup))

# Close the WebDriver
driver.quit()

# Create a pandas DataFrame with the scraped data
df = pd.DataFrame(match_history)

# Save the DataFrame to an Excel file
df.to_excel("match_history.xlsx", index=False)

# Look at winrates by adding a column of who won
# Step 1: Create a new column 'Outcome'
def get_winner(score, team1, team2):
    s1, s2 = map(int, score.split(" - "))
    if s1 > s2:
        return team1
    if s1 < s2:
        return team2
    else:
        return "Draw"

df["Outcome"] = df[["Score", "Team 1", "Team 2"]].apply(lambda row: get_winner(row["Score"], row["Team 1"], row["Team 2"]), axis=1)

# Step 2: Create a new DataFrame 'bo1s' for rows with a score superior to 3 to account for matches that are either BO1 or BO5
def is_score_superior_to_3(score):
    s1, s2 = map(int, score.split(" - "))
    return s1 > 3 or s2 > 3

bo1s = df[df["Score"].apply(is_score_superior_to_3)].copy()
df = df[~df["Score"].apply(is_score_superior_to_3)]


In [ ]:
df

In [ ]:
# Create a new DataFrame called 'winrate' with a column 'Team' containing all unique teams
winrate = pd.DataFrame({'Team': df[['Team 1', 'Team 2']].stack().unique()})

# Calculate the total matches played by each team
total_matches = df[['Team 1', 'Team 2']].stack().value_counts().rename_axis('Team').reset_index(name='Total Matches')
winrate = winrate.merge(total_matches, on='Team', how='left')

# Calculate the wins by counting each team in the 'Outcome' column
wins = df['Outcome'].value_counts().rename_axis('Team').reset_index(name='Wins')
winrate = winrate.merge(wins, on='Team', how='left')

# Fill NaN values in the 'Wins' column with 0
winrate['Wins'].fillna(0, inplace=True)

# Calculate the defeats
winrate['Defeats'] = winrate['Total Matches'] - winrate['Wins']

# Calculate the winrate
winrate['Winrate'] = winrate['Wins'] / winrate['Total Matches']


In [ ]:
winrate

In [ ]:
# Filter the winrate DataFrame to only include teams that have played at least 50 matches
filtered_winrate = winrate[winrate['Total Matches'] >= 50]

# Plot the distribution of winrates using a histogram

plt.hist(filtered_winrate['Winrate'], bins=20, density=True, alpha=0.6, edgecolor='black')
plt.xlabel('Winrate')
plt.ylabel('Density')
plt.title('Winrate Distribution')
plt.show()

# Alternatively, plot the distribution of winrates using a KDE plot
kde = gaussian_kde(filtered_winrate['Winrate'])
x = np.linspace(0, 1, 100)
plt.plot(x, kde(x), label='Winrate KDE')
plt.xlabel('Winrate')
plt.ylabel('Density')
plt.title('Winrate Distribution (KDE)')
plt.legend()
plt.show()
